## Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
from IPython.display import display
import datetime
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Using device: {device}")


✓ Using device: cpu


## Imports

In [3]:
from embeddings.clap import CLAPWrapper
from effects.fx import FXChainFactory
from llms.llmclient import LLMClient
from prompts.prompt import Prompt, PromptFactory
from configurations.config import Config
from training.loss import refine_with_directional_loss
from utilities.audio_processing import play_audio, play_audio_from_file
from training.trainer import ParameterEngine
from configurations.config import *
from utilities.fx_processing import fx_initial_params_to_tensor
from effects.fx import ALL_PARAM_RANGES, llm_params_tensor_example, llm_params_dict_example
from utilities.audio_processing import save_audio_batch

## Load Models

In [ ]:
from fxsearcher.fxsearcher import fxsearcher
clap = CLAPWrapper(device=device)

/Users/milanliessens/miniconda3/envs/pwfx/lib/python3.10/site-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/TensorShape.cpp:4383.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Load our best checkpoint in the paper.
The checkpoint is already downloaded
Load Checkpoint...
logit_scale_a 	 Loaded
logit_scale_t 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_real.weight 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_imag.weight 	 Loaded
audio_branch.logmel_extractor.melW 	 Loaded
audio_branch.bn0.weight 	 Loaded
audio_branch.bn0.bias 	 Loaded
audio_branch.patch_embed.proj.weight 	 Loaded
audio_branch.patch_embed.proj.bias 	 Loaded
audio_branch.patch_embed.norm.weight 	 Loaded
audio_branch.patch_embed.norm.bias 	 Loaded
audio_branch.layers.0.blocks.0.norm1.weight 	 Loaded
audio_branch.layers.0.blocks.0.norm1.bias 	 Loaded
audio_branch.layers.0.blocks.0.attn.relative_position_bias_table 	 Loaded
audio_branch.layers.0.blocks.0.attn.qkv.weight 	 Loaded
audio_branch.layers.0.blocks.0.attn.qkv.bias 	 Loaded
audio_branch.layers.0.blocks.0.attn.proj.weight 	 Loaded
audio_branch.layers.0.blocks.0.attn.proj.bias 	 Loaded
audio_branch.layers.0.blocks.0.norm2.we

## Load Audio

In [4]:
# Load audio file
try:
    # Try Colab file upload
    from google.colab import files
    print("📤 Upload an audio file (.wav, .mp3):")
    uploaded = files.upload()
    audio_filename = list(uploaded.keys())[0]
except:
    # Local environment - specify your audio file here
    audio_filename = "../data/audio/piano.wav"
    print(f"📁 Using local audio file: {audio_filename}")

from utilities.audio_processing import load_and_preprocess_audio
audio = load_and_preprocess_audio(audio_filename, device)

print(f"✓ Audio loaded: shape={audio.shape}")
# Uncomment to play audio:
print("\n🎵 Original audio:")
play_audio(audio.cpu())

📁 Using local audio file: ../data/audio/piano.wav
✓ Audio loaded: shape=torch.Size([1, 2, 441000])

🎵 Original audio:


## Set Instructions

In [5]:
llmclient = LLMClient()
fx_chain_factory = FXChainFactory()
fx_chain = fx_chain_factory.create_fx_chain(sample_rate=44100, device=device)

text_anchor = "This sound is dark"
text_target = "This sound is bright"
task = "Make the sound brighter"
instruction = "This is piano music. " + task

prompt = PromptFactory.LLM_PARAMETER_INITIALIZATION_PROMPT_DASP(fx_chain, instruction)
prompt_pedalboard = PromptFactory.LLM_PARAMETER_INITIALIZATION_PROMPT_PEDALBOARD(instruction)

✓ LLM client ready
✓ FX chain created: 49 parameters


In [6]:
parameter_engine = ParameterEngine()

In [ ]:
from fxsearcher.fxsearcher_og import fxsearcher_og